The SQL code below builds a clean analysis workflow for an e-commerce dataset by exploring customers, orders, products, and website activity, then creating views that focus only on paid orders. The main goal is to organize the raw data into useful tables that make business analysis easier, such as tracking revenue by product category, identifying top customers each month, measuring how customer spending grows over time, and calculating the time between customer orders. The outcome is a set of structured SQL views and queries that transform raw transactional data into clear insights for reporting, customer behavior analysis, and future dashboard or Tableau use.

In [20]:
sql_query = """

SELECT TOP(3) * FROM SimpleInterview.dbo.customers
SELECT TOP(3) * FROM SimpleInterview.dbo.daily_site_visits
SELECT TOP(5) * FROM SimpleInterview.dbo.order_items
SELECT TOP(3) * FROM SimpleInterview.dbo.orders
SELECT TOP(3) * FROM SimpleInterview.dbo.products

--Create a view that returns one row per order line item
--(each product on an order), but only for paid orders.

 SELECT COUNT(DISTINCT (O.order_id)) AS Disticnt_Count
 FROM SimpleInterview.dbo.orders AS O
 INNER JOIN SimpleInterview.dbo.order_items AS OI
 ON O.order_id = OI.order_id
 INNER JOIN SimpleInterview.dbo.customers AS C
 ON O.customer_id = C.customer_id
 INNER JOIN SimpleInterview.dbo.products AS P
 ON OI.product_id = P.product_id
 WHERE O.payment_status = 'paid'

 SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN C.region IS NULL THEN 1 ELSE 0 END)    AS null_region_rows,
    SUM(CASE WHEN P.category IS NULL THEN 1 ELSE 0 END)  AS null_category_rows
FROM SimpleInterview.dbo.orders AS O
INNER JOIN SimpleInterview.dbo.order_items AS OI
    ON O.order_id = OI.order_id
INNER JOIN SimpleInterview.dbo.customers AS C
    ON O.customer_id = C.customer_id
INNER JOIN SimpleInterview.dbo.products AS P
    ON OI.product_id = P.product_id
WHERE O.payment_status = 'paid';

USE SimpleInterview;
GO

CREATE OR ALTER VIEW dbo.v_paid_order_lines AS

SELECT
 O.order_id,
 O.order_date,
 O.customer_id,
 O.channel,
 C.region,
 OI.product_id,
 OI.quantity,
 OI.unit_price,
 OI.line_revenue,
 P.category
 FROM SimpleInterview.dbo.orders AS O
 INNER JOIN SimpleInterview.dbo.order_items AS OI
 ON O.order_id = OI.order_id
 INNER JOIN SimpleInterview.dbo.customers AS C
 ON O.customer_id = C.customer_id
 INNER JOIN SimpleInterview.dbo.products AS P
 ON OI.product_id = P.product_id
 WHERE O.payment_status = 'paid'

GO

SELECT COUNT(*) Total_Rows FROM SimpleInterview.dbo.v_paid_order_lines
SELECT COUNT(DISTINCT (order_id)) AS Total_UniqueIDs FROM SimpleInterview.dbo.v_paid_order_lines

--For each month, find the top 3 product categories by total revenue.

SELECT * FROM SimpleInterview.dbo.v_paid_order_lines

WITH monthly_category AS (
    SELECT
        DATEFROMPARTS(YEAR(order_date), MONTH(order_date), 1) AS month_start,
        category,
        SUM(line_revenue) AS total_revenue
    FROM SimpleInterview.dbo.v_paid_order_lines
    GROUP BY
        DATEFROMPARTS(YEAR(order_date), MONTH(order_date), 1),
        category
),

ranked AS (

    SELECT
        month_start,
        category,
        total_revenue,
        DENSE_RANK() OVER (
            PARTITION BY month_start
            ORDER BY total_revenue DESC
        ) AS revenue_rank
    FROM monthly_category

)

SELECT
    month_start,
    category,
    total_revenue,
    revenue_rank
FROM ranked
WHERE revenue_rank <= 3
ORDER BY month_start, revenue_rank, category;

--For each month, find the top 5 customers by total revenue

SELECT * FROM SimpleInterview.dbo.v_paid_order_lines

WITH Customer_MonthTotalRev AS (

SELECT
DATEFROMPARTS(YEAR(order_date), MONTH(order_date), 1) AS month_start,
customer_id AS Customer,
SUM(line_revenue) AS Total_Revenue
FROM SimpleInterview.dbo.v_paid_order_lines
GROUP BY customer_id, DATEFROMPARTS(YEAR(order_date), MONTH(order_date), 1)

),

Ranked_CustRev AS

(
SELECT
month_start,
Customer,
Total_Revenue,
DENSE_RANK() OVER(PARTITION BY month_start ORDER BY Total_Revenue DESC) AS RankedCust
FROM Customer_MonthTotalRev
)

SELECT
month_start,
Customer,
Total_Revenue,
RankedCust
FROM Ranked_CustRev
WHERE RankedCust < 6

--“For each customer, how does their total spend accumulate over time?”

WITH Cust_Spend_OverTime AS (

SELECT
DATEFROMPARTS(YEAR(order_date), MONTH(order_date), 1) AS month_start,
customer_id AS Customer,
SUM(line_revenue) AS Total_Revenue
FROM SimpleInterview.dbo.v_paid_order_lines
GROUP BY customer_id, DATEFROMPARTS(YEAR(order_date), MONTH(order_date), 1)

)

SELECT
month_start,
Customer,
Total_Revenue,
SUM(Total_Revenue) OVER(PARTITION BY Customer ORDER BY month_start) AS Running_Total
FROM Cust_Spend_OverTime

--cumulative revenue per customer order-by-order

WITH Cum_Rev_OBO AS (

SELECT
order_id,
order_date,
customer_id,
SUM(line_revenue) AS Total_Revenue
FROM SimpleInterview.dbo.v_paid_order_lines
GROUP BY
order_id,
order_date,
customer_id
)

SELECT
order_id,
order_date,
customer_id,
SUM(Total_Revenue) OVER(PARTITION BY customer_id ORDER BY order_date, order_id) AS Runn_Total
FROM Cum_Rev_OBO
ORDER BY customer_id, order_date, order_id

--For each customer’s orders, calculate: previous order date, days since previous order

SELECT * FROM SimpleInterview.dbo.v_paid_order_lines

WITH Prev_day AS (

SELECT
order_id,
customer_id,
order_date,
SUM(line_revenue) AS Total_Revenue
FROM SimpleInterview.dbo.v_paid_order_lines
GROUP BY customer_id, order_date, order_id

),

Prev_Order_Date AS (

SELECT
order_id,
customer_id,
order_date,
LAG(order_date) OVER(PARTITION BY customer_id ORDER BY order_date, order_id) AS prev_order_date,
Total_Revenue
FROM Prev_day

)

SELECT
order_id,
customer_id,
order_date,
prev_order_date,
Total_Revenue,
COALESCE(DATEDIFF(day, prev_order_date, order_date), 0) AS DayDiff_between_Orders
FROM Prev_Order_Date

--build one clean “customer-order features” table from SQL

SELECT * FROM SimpleInterview.dbo.v_paid_order_lines

USE SimpleInterview;
GO

CREATE OR ALTER VIEW dbo.v_customer_order_features
AS
WITH CTE_FirstTable AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        region,
        channel,
        SUM(line_revenue) AS Total_Revenue
    FROM SimpleInterview.dbo.v_paid_order_lines
    GROUP BY customer_id, order_date, order_id, region, channel
),
CTE_SecondTable AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        Total_Revenue,
        region,
        channel,
        LAG(order_date) OVER(PARTITION BY customer_id ORDER BY order_date, order_id) AS prev_order_date
    FROM CTE_FirstTable
),
CTE_ThirdTable AS (
    SELECT
        order_id,
        customer_id,
        order_date,
        Total_Revenue,
        prev_order_date,
        region,
        channel,
        COALESCE(DATEDIFF(day, prev_order_date, order_date), 0) AS days_since_prev_order
    FROM CTE_SecondTable
)
SELECT
    order_id,
    customer_id,
    order_date,
    Total_Revenue,
    prev_order_date,
    days_since_prev_order,
    region,
    channel,
    SUM(Total_Revenue) OVER(
        PARTITION BY customer_id
        ORDER BY order_date, order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_revenue
FROM CTE_ThirdTable;
GO

SELECT *
FROM SimpleInterview.dbo.v_customer_order_features
ORDER BY customer_id, order_date, order_id;

"""


The Python code below takes a customer orders dataset, cleans and prepares it for analysis, creates new customer behavior features, and calculates business KPIs such as total revenue, total orders, repeat order rate, and average order value. The goal is to better understand customer purchasing patterns over time, by region, and by sales channel, while also segmenting customers based on recency. It then uses a linear regression model to predict order revenue from customer and order features. The outcome is an end-to-end analytics workflow that combines data cleaning, KPI reporting, customer behavior analysis, and basic predictive modeling for business insights.

In [21]:
pip install scikit-learn

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import root_mean_squared_error, r2_score, accuracy_score, confusion_matrix
from tensorflow import keras
from tensorflow.keras import layers

In [23]:
#Uploading the csv table
from google.colab import files
uploaded = files.upload()

Saving customer_order_features.csv to customer_order_features (1).csv


In [24]:
df = pd.read_csv("customer_order_features.csv")
df.head(2)

,708,1,2025-08-28,463.95,NULL,0,Midwest,web,463.95.1
0,29,1,2025-09-19,1593.96,2025-08-28,22,Midwest,store,2057.91
1,92,1,2025-11-13,369.69,2025-09-19,55,Midwest,web,2427.60


In [25]:
#Pandas is not reading the table's headers
columns_name = ["order_id", "customer_id", "order_date", "Total_Revenue", "prev_order_date", "days_since_prev_order", "region", "channel", "running_revenue"]
df = pd.read_csv("customer_order_features.csv", header=None, names=columns_name)
df.head(2)

,order_id,customer_id,order_date,Total_Revenue,prev_order_date,days_since_prev_order,region,channel,running_revenue
0,708,1,2025-08-28,463.95,NaN,0,Midwest,web,463.95
1,29,1,2025-09-19,1593.96,2025-08-28,22,Midwest,store,2057.91


In [26]:
df.isna().sum()

,0
order_id,0
customer_id,0
order_date,0
Total_Revenue,0
prev_order_date,169
days_since_prev_order,0
region,0
channel,0
running_revenue,0


In [27]:
df["prev_order_date"] = df["prev_order_date"].replace("NULL", np.nan)
df.loc[df["prev_order_date"].eq("NULL"),["prev_order_date"]].sum()

,0
prev_order_date,0


In [28]:
#Convert datetimes
df["order_date"] = pd.to_datetime(df["order_date"])
df["prev_order_date"] = pd.to_datetime(df["prev_order_date"])

df[["order_date", "prev_order_date"]].dtypes

,0
order_date,datetime64[ns]
prev_order_date,datetime64[ns]


In [29]:
#Checking duplicate and unique values
unique_order_id = df["order_id"].is_unique
print(unique_order_id)

#Corrobotating no duplicates
order_id_duplicates = df.duplicated(subset=["order_id"]).sum()
print(order_id_duplicates)

True
0


In [30]:
#Creating new columns for analysis

# Order number per customer (1st order, 2nd order, etc.)
df["order_number_per_customer"] = df.groupby("customer_id")["order_id"].cumcount() + 1

# Prior revenue (what did they spend last time?)
df["money_spent_in_previous_order"] = df.groupby("customer_id")["Total_Revenue"].shift(1)

#Running average spend
df["average_money_spend_by_customer"] = df["running_revenue"] / df["order_number_per_customer"]

#Counting how many times a customer repeats ordering
df["repeating_order_by_customer"] = (df["order_number_per_customer"] > 1).astype(int)

df.head()

,order_id,customer_id,order_date,Total_Revenue,prev_order_date,days_since_prev_order,region,channel,running_revenue,order_number_per_customer,money_spent_in_previous_order,average_money_spend_by_customer,repeating_order_by_customer
0,708,1,2025-08-28,463.95,NaT,0,Midwest,web,463.95,1,NaN,463.950,0
1,29,1,2025-09-19,1593.96,2025-08-28,22,Midwest,store,2057.91,2,463.95,1028.955,1
2,92,1,2025-11-13,369.69,2025-09-19,55,Midwest,web,2427.60,3,1593.96,809.200,1
3,194,3,2025-10-10,235.44,NaT,0,West,mobile,235.44,1,NaN,235.440,0
4,185,3,2025-11-07,918.45,2025-10-10,28,West,web,1153.89,2,235.44,576.945,1


In [31]:
#KPI calculations

#Total Revenue
total_revenue = df["Total_Revenue"].sum()

#Total amount of unique orders
total_orders = df["order_id"].nunique()

#Total amount of unique customers
total_customers = df["customer_id"].nunique()

#Average order value
average_order_value = round(total_revenue/total_orders, 2)

#The rate of a customer repeating ordering
repeat_order_rate = round(df["repeating_order_by_customer"].mean(), 2)

#The average number of orders per customer
average_orders_per_customer = round(total_orders / total_customers,2)

#The average revenue per customer
average_revenue_per_customer = round(total_revenue / total_customers,2)

#The average number of days between orders
average_days_between_orders_per_customer = df.loc[df["repeating_order_by_customer"].eq(1),["repeating_order_by_customer"]].mean()

KPI_Calculations_Table = pd.DataFrame({
    "KPI": [
        "total_revenue",
        "total_orders",
        "total_customers",
        "average_order_value",
        "repeat_order_rate",
        "average_orders_per_customer",
        "average_revenue_per_customer",
        "average_days_between_orders_per_customer",
    ],
    "Value": [
        total_revenue,
        total_orders,
        total_customers,
        average_order_value,
        repeat_order_rate,
        average_orders_per_customer,
        average_revenue_per_customer,
        average_days_between_orders_per_customer
       ]
})

KPI_Calculations_Table


,KPI,Value
0,total_revenue,420352.18
1,total_orders,682
2,total_customers,169
3,average_order_value,616.35
4,repeat_order_rate,0.75
5,average_orders_per_customer,4.04
6,average_revenue_per_customer,2487.29
7,average_days_between_orders_per_customer,repeating_order_by_customer 1.0 dtype: float64


In [32]:
#KPI reporting by month (trend over time)
df["order_month"] = df["order_date"].dt.to_period("M").dt.to_timestamp()

monthly = (df.groupby("order_month").agg(
    revenue=("Total_Revenue", "sum"),
    orders=("order_id", "nunique"),
    customers=("customer_id", "nunique"),
    repeat_order_rate=("repeating_order_by_customer", lambda x: round(x.mean(), 2)),
    aov = ("Total_Revenue", lambda x: round(x.mean(), 1))
    )).reset_index().sort_values("order_month")


#KPI reporting by region
region = (df.groupby("region").agg(
    revenue=("Total_Revenue", "sum"),
    orders=("order_id", "nunique"),
    customer=("customer_id", "nunique"),
    repeat_order_rate=("repeating_order_by_customer", lambda x: round(x.mean(), 2)),
    aov = ("Total_Revenue", lambda x: round(x.mean(), 2))
)).reset_index().sort_values("region")


#KPI reporting by channel
channel = (df.groupby("channel").agg(
    revenue=("Total_Revenue", "sum"),
    orders=("order_id", "nunique"),
    customer=("customer_id", "nunique"),
    repeat_order_rate=("repeating_order_by_customer", lambda x: round(x.mean(), 2)),
    aov = ("Total_Revenue", lambda x: round(x.mean(), 2))
)).reset_index().sort_values("channel")




In [33]:
monthly.head(10)

,order_month,revenue,orders,customers,repeat_order_rate,aov
0,2025-08-01,48522.41,88,69,0.22,551.4
1,2025-09-01,56752.11,90,67,0.59,630.6
2,2025-10-01,54211.49,88,64,0.74,616.0
3,2025-11-01,80631.46,131,88,0.84,615.5
4,2025-12-01,87906.17,138,90,0.93,637.0
5,2026-01-01,92328.54,147,88,0.94,628.1


In [34]:
region.head()

,region,revenue,orders,customer,repeat_order_rate,aov
0,Midwest,47634.69,87,26,0.70,547.53
1,Northeast,102029.20,159,46,0.71,641.69
2,South,108198.37,183,40,0.78,591.25
3,West,162489.92,253,57,0.77,642.25


In [35]:
channel.head()

,channel,revenue,orders,customer,repeat_order_rate,aov
0,mobile,141172.54,243,122,0.74,580.96
1,store,85172.91,132,83,0.75,645.25
2,web,194006.73,307,135,0.76,631.94


In [36]:
#KPI Recency score
as_of_date = df["order_date"].max() + pd.Timedelta(days=1)

recency_table = (df.groupby("customer_id")["order_date"].max().to_frame("last_order_date"))
recency_table["recency_days"] = (as_of_date - recency_table["last_order_date"]).dt.days

recency_table["Recency_Score"] = pd.qcut(recency_table["recency_days"], 5, labels=[5,4,3,2,1]) #Recency score: lower recency_days is better, so reverse labels

recency_table["Recency_Score"] = recency_table["Recency_Score"].astype(int)

def segment(customer_score):
  R = customer_score["Recency_Score"]
  if R >= 4:
    return "High Recency"
  if R <= 2:
    return "Low Recency"
  else:
    return "Hibernating / Lost"

recency_table['segment'] =  recency_table.apply(segment, axis=1)

recency_table["segment"].value_counts()


,count
segment,
High Recency,69
Low Recency,68
Hibernating / Lost,32


In [37]:
df.head()

,order_id,customer_id,order_date,Total_Revenue,prev_order_date,days_since_prev_order,region,channel,running_revenue,order_number_per_customer,money_spent_in_previous_order,average_money_spend_by_customer,repeating_order_by_customer,order_month
0,708,1,2025-08-28,463.95,NaT,0,Midwest,web,463.95,1,NaN,463.950,0,2025-08-01
1,29,1,2025-09-19,1593.96,2025-08-28,22,Midwest,store,2057.91,2,463.95,1028.955,1,2025-09-01
2,92,1,2025-11-13,369.69,2025-09-19,55,Midwest,web,2427.60,3,1593.96,809.200,1,2025-11-01
3,194,3,2025-10-10,235.44,NaT,0,West,mobile,235.44,1,NaN,235.440,0,2025-10-01
4,185,3,2025-11-07,918.45,2025-10-10,28,West,web,1153.89,2,235.44,576.945,1,2025-11-01


In [38]:
#Linear Regression
df_model = df[["Total_Revenue", "region", "channel", "order_number_per_customer", "average_money_spend_by_customer"]]

df_model = pd.get_dummies(df_model, columns=["region", "channel"], drop_first=False, dtype=int)

# -------- Define features (X) and target (y) --------
X = df_model.drop("Total_Revenue", axis=1)
y = df_model["Total_Revenue"]

# -------- Split into training and test sets --------
# test_size=0.2 -> 20% for testing, 80% for training
# random_state=42 -> makes the split reproducible
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# -------- Create the linear regression model --------
model = LinearRegression()

# -------- Train (fit) the model --------
model.fit(X_train, y_train)

# -------- 7. Make predictions on the test set --------
y_pred = model.predict(X_test)

# Show a few example predictions vs actual values
print("\nSample predictions vs actual values:")
for true_val, pred_val in list(zip(y_test[:5], y_pred[:5])):
    print(f"Actual: {true_val:.2f}  |  Predicted: {pred_val:.2f}")

# -------- Evaluate the model --------
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(rmse)
print(r2)


Sample predictions vs actual values:
Actual: 435.12  |  Predicted: 774.25
Actual: 702.88  |  Predicted: 283.00
Actual: 206.70  |  Predicted: 143.34
Actual: 475.82  |  Predicted: 552.86
Actual: 1043.60  |  Predicted: 802.49
357.533312918645
0.3930854957524129
